# Day 2 · Embeddings by hand — summary

**Question:** do frames and their transcript land near each other in embedding space, on my footage?
**Answer:** yes, decisively — but most of the signal is topic-level, not temporal.

## Setup
`lecture01`, first 40 min. 20 (frame, caption) pairs, stratified random, seed 42, **zero replacements**.
Captions = YouTube auto-subs, every cue overlapping ±5s, rule fixed before measuring.
All numbers scored as separation from a **100-derangement shuffled baseline**, in units of that baseline's SD — absolute cosines are meaningless under the modality gap.

## Results — CLIP ViT-B/32

| level | mean cosine | vs previous level |
|---|---|---|
| matched | 0.2673 | **9.5 SD** above shuffled |
| shuffled (same video) | 0.2310 ± 0.0038 | **13.8 SD** above cross-video |
| cross-video (ocean podcast) | 0.1784 | — |

recall@1: 0.40 image→text, 0.40 text→image, chance 0.05 · median rank 4.0 of 20, chance 10.5

**Predicted 0.75, measured 0.40.** The gap is the day's lesson, not a failure.

## Checkpoints — three levels each

| | CLIP B/32 | SigLIP B/16 | SigLIP 2 B/16 |
|---|---|---|---|
| within-video (matched vs shuffled) | **9.5 SD** | 9.3 SD | 7.2 SD |
| topic-level (shuffled vs cross) | 13.8 SD | 12.1 SD | **21.7 SD** |
| ratio within/topic | 0.69 | 0.77 | **0.33** |
| recall@1 i→t / t→i | 0.40 / 0.40 | 0.25 / 0.45 | 0.30 / 0.25 |
| median rank | 4.0 | 5.0 | **3.0** |

Matched means are 0.267 / 0.073 / 0.126 — **not comparable across objectives**. The SD ruler is what makes these three rows mean anything.

## Findings

1. **Alignment is real.** 9.5 SD is not luck. This justifies building the multimodal half at all.
2. **The easy discrimination is topic; the hard one is time.** Separating a lecture from an ocean podcast is ~1.5× easier than separating minute 12 from minute 31 of the same lecture. Ravid already knows which video she's in — the project lives entirely in the harder gap.
3. **SigLIP 2 is best at the discrimination the task doesn't need.** Ratio 0.33 vs CLIP's 0.69. Best median rank (3.0) though — genuinely mixed. Chosen for the BSc project on multilingual support, *not* on these numbers.
4. **Recall@1 and median rank disagree.** 0.40 sounds like "half-works"; median rank 2 (t→i) says "orders well, rarely nails the top slot." Retrieval-precision problem, not an alignment problem → this is the argument for a reranker in W3.
5. **Two captions act as hubs**, winning six rows they don't own. Why image→text is worse than text→image. Watch hubness when the pool grows in W2.
6. **The 77-token wall.** 873-word chunk → 1121 tokens → 92% discarded silently. cosine(full, first-77) = **1.000000**: the model never saw the rest. Not degradation — structural impossibility. Token ratio ≈ 1.28 tok/word on this transcript.

## Decisions made today

- **Two vector spaces, two jobs.** `text_dense` (sentence embedder, full chunk, retrieval) and `clip_text` (≤77 tok, short spans, cross-modal only). Never compared to each other.
- **Every similarity is a delta** against a stated baseline. No absolute thresholds anywhere in this project.
- **Derangements, not permutations**, for shuffled baselines — fixed points smuggle matched pairs into the baseline and shrink the delta.
- Caption rule (±5s) was **not** revised after seeing 0.40. Window sweep reported as sensitivity analysis, not a search.

## Caveats

n=20, one video, one 40-min trim. Enough to establish alignment exists; **not** enough to rank two good checkpoints or trust a 1-item recall@1 difference. transformers 5.14.1 returns tower outputs from `get_*_features` — this notebook calls the projections explicitly.

## Carried into Week 2

- Re-test the within/cross ratio at 100 pairs × 3 videos. Does SigLIP 2's 0.33 hold?
- Watch caption hubness as the candidate pool grows from 20 to thousands.
- Fix the tie convention (irrelevant at n=20, zero exact ties) before the pool scales.

In [1]:
# --- 0 · setup -------------------------------------------------------------
# Everything the notebook needs to survive a kernel restart lives here.
import json, random, re, subprocess
from pathlib import Path

ROOT  = Path.cwd().parent                      # notebooks/ -> repo root
VIDEO = "lecture01"
SRC   = ROOT / f"data/raw/{VIDEO}/{VIDEO}_t0-2400.mp4"
VTT   = ROOT / f"data/raw/{VIDEO}/{VIDEO}.en.vtt"
PAIRS = ROOT / f"data/pairs/{VIDEO}"
PAIRS.mkdir(parents=True, exist_ok=True)

SEED = 42        # written down = reproducible
DUR  = 2400      # seconds of the trimmed video
N    = 20        # pairs
PAD  = 5.0       # caption half-window, in seconds — the pre-registered rule

assert SRC.exists(), f"missing {SRC}"
assert VTT.exists(), f"missing {VTT}"
print(ROOT)

c:\Users\tawil\Portfolio\12-weeks\video-rag


In [2]:
# --- 1 · timestamps --------------------------------------------------------
# Stratified random: split the video into N equal buckets, take one random point
# inside each. Covers the whole lecture without the clustering pure random gives,
# and without me choosing the moments (which would be cherry-picking).
random.seed(SEED)
bucket = DUR / N
TS = [round(i * bucket + random.uniform(0.15, 0.85) * bucket, 2) for i in range(N)]
print(TS)

[71.71, 140.1, 281.1, 396.75, 559.86, 674.84, 812.94, 865.3, 1013.44, 1100.5, 1236.37, 1380.45, 1460.23, 1594.7, 1752.59, 1863.78, 1956.52, 2107.5, 2245.99, 2298.55]


In [3]:
# --- 2 · frames ------------------------------------------------------------
# One frame per timestamp. Filenames carry the timestamp, same rule as the rest
# of the pipeline. Skips work already done, so a rerun is free.
def frame_name(i, t):
    return f"{i:02d}_{int(t // 60):02d}-{int(t % 60):02d}.jpg"

made = 0
for i, t in enumerate(TS):
    dst = PAIRS / frame_name(i, t)
    if dst.exists():
        continue
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", str(t),
                    "-i", str(SRC), "-frames:v", "1", "-q:v", "2", str(dst)],
                   check=True)
    made += 1

print(f"{made} extracted, {N - made} already present -> {PAIRS}")
# Inspection log (Day 2): all 20 usable. Five are semantically thin —
# 00 title, 02/03 phone imagery, 12/15 section-title cards — kept as drawn.
# No blur, no black frames, no transitions. Zero replacements.

0 extracted, 20 already present -> c:\Users\tawil\Portfolio\12-weeks\video-rag\data\pairs\lecture01


In [4]:
# --- 3 · captions ----------------------------------------------------------
# YouTube auto-captions, deliberately: imperfect ASR, roughly the quality Whisper
# will produce tomorrow. Hand-written captions would be cleaner than anything the
# pipeline can make, which would inflate today's number and make it untransferable.
#
# Auto-caption VTT is a ROLLING format: each cue repeats the tail of the previous
# one. Naive parsing duplicates every word two or three times.
import webvtt

cues = []
for c in webvtt.read(str(VTT)):
    txt = re.sub(r"<[^>]+>", "", c.text).replace("\n", " ").strip()
    txt = re.sub(r"\s+", " ", txt)
    if not txt:
        continue
    if cues and (txt in cues[-1][2] or cues[-1][2].endswith(txt)):
        continue                                  # wholly contained in the last cue
    if cues and txt.startswith(cues[-1][2]):      # cue grew from the previous one
        txt = txt[len(cues[-1][2]):].strip()
        if not txt:
            continue
    cues.append((c.start_in_seconds, c.end_in_seconds, txt))

def caption(t, pad=PAD):
    """Every cue overlapping [t-pad, t+pad], joined. `pad` is the rule under test.

    The trim started at 0s, so caption times and video times are the same. A
    non-zero trim offset would have to be subtracted here.
    """
    lo, hi = t - pad, t + pad
    return " ".join(x for s, e, x in cues if e >= lo and s <= hi).strip()

print(f"{len(cues)} cues parsed\n")
for i, t in enumerate(TS):
    print(f"{i:02d}  {int(t//60):02d}:{int(t%60):02d}  {caption(t)}")

1569 cues parsed

00  01:11  is, but also remember how we got there. And I think it's really easy for us to become desensitized actually with a lot of the state of the art with AI because there are so many amazing things
01  02:20  this revolution expand also into natural language as well. So, in 2022, we had ChatGPT-3 released and launched to the public followed shortly by GPT-4 in 2023.
02  04:41  is running entirely on airplane mode. There's no Wi-Fi connected to this phone. This is entirely offline. Everything that's being computed is running on the device itself. Now,
03  06:36  what have we seen? Right. So, this is an end-to-end audio model. It means that the latency is extremely low is what you'll probably notice there because number one, you don't send anything to the cloud, so you don't have to wait for internet times. Everything runs on
04  09:19  focuses on the use of neural networks, specifically deep neural networks, to do this task of learning from data to inform future d

In [5]:
# --- 4 · freeze the pairs --------------------------------------------------
# The artifact of the slow hour. Everything downstream reads this file, so the
# notebook is rerunnable without redoing any of the manual work.
# pad is passed explicitly: the file should record the rule, not inherit a default.
pairs = [{"i": i, "ts": t, "frame": frame_name(i, t), "caption": caption(t, pad=PAD)}
         for i, t in enumerate(TS)]
(PAIRS / "pairs.json").write_text(json.dumps(pairs, indent=2))
print(f"wrote {len(pairs)} pairs at pad={PAD}s -> {PAIRS / 'pairs.json'}")

wrote 20 pairs at pad=5.0s -> c:\Users\tawil\Portfolio\12-weeks\video-rag\data\pairs\lecture01\pairs.json


In [6]:
# --- 5 · encode ------------------------------------------------------------
# Two separate transformers. The image tower cuts the frame into 32x32 patches and
# runs a ViT over them; the text tower runs over tokens. Their outputs live in
# UNRELATED spaces (768-d and 512-d). Each *_projection is one learned linear layer
# into the shared 512-d space — that projection is the whole of "one space for
# pixels and words", and it is what contrastive pre-training actually trained.
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL_ID).eval()
proc  = CLIPProcessor.from_pretrained(MODEL_ID)

pairs = json.loads((PAIRS / "pairs.json").read_text())
imgs  = [Image.open(PAIRS / p["frame"]).convert("RGB") for p in pairs]

def encode_text(texts):
    with torch.no_grad():
        out = model.text_model(**proc(text=texts, return_tensors="pt",
                                      padding=True, truncation=True))
        v = model.text_projection(out.pooler_output)
    return v / v.norm(dim=-1, keepdim=True)

# Frames never change across the experiments below — encode once, reuse.
with torch.no_grad():
    vout = model.vision_model(**proc(images=imgs, return_tensors="pt"))
    img  = model.visual_projection(vout.pooler_output)
img = img / img.norm(dim=-1, keepdim=True)   # unit length -> dot product IS cosine

txt = encode_text([p["caption"] for p in pairs])
print(img.shape, txt.shape)

c:\Users\tawil\Portfolio\12-weeks\video-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 22676.23it/s]


torch.Size([20, 512]) torch.Size([20, 512])


In [7]:
# --- 6 · the matrix --------------------------------------------------------
# S[i][j] = frame i vs caption j. Rows = frames, cols = captions, diagonal = truth.
# Getting the orientation backwards silently transposes the whole result.
S = img @ txt.T

torch.set_printoptions(precision=3, sci_mode=False, linewidth=200)
print(S)
print(f"\nrange {S.min():.3f} .. {S.max():.3f}   diag mean {S.diag().mean():.3f}   "
      f"off-diag mean {(S.sum() - S.diag().sum()).item() / (N*N - N):.3f}")
# Every entry sits in a narrow band well below 1.0, including correct pairs.
# That is the modality gap (Liang et al. 2022) — never threshold a cosine here.

tensor([[0.230, 0.230, 0.139, 0.215, 0.283, 0.224, 0.239, 0.270, 0.186, 0.181, 0.227, 0.205, 0.222, 0.218, 0.215, 0.264, 0.232, 0.228, 0.156, 0.189],
        [0.254, 0.316, 0.172, 0.258, 0.255, 0.226, 0.208, 0.249, 0.206, 0.194, 0.229, 0.233, 0.238, 0.246, 0.223, 0.239, 0.233, 0.208, 0.205, 0.247],
        [0.212, 0.223, 0.259, 0.253, 0.224, 0.232, 0.231, 0.230, 0.198, 0.181, 0.220, 0.192, 0.221, 0.220, 0.235, 0.227, 0.199, 0.185, 0.194, 0.186],
        [0.213, 0.227, 0.243, 0.248, 0.235, 0.210, 0.228, 0.215, 0.201, 0.172, 0.247, 0.206, 0.213, 0.218, 0.204, 0.212, 0.206, 0.206, 0.229, 0.175],
        [0.231, 0.227, 0.169, 0.236, 0.282, 0.210, 0.239, 0.260, 0.190, 0.196, 0.241, 0.197, 0.231, 0.226, 0.235, 0.255, 0.235, 0.218, 0.178, 0.166],
        [0.214, 0.201, 0.159, 0.241, 0.223, 0.323, 0.201, 0.242, 0.200, 0.215, 0.207, 0.186, 0.218, 0.193, 0.207, 0.218, 0.232, 0.202, 0.172, 0.156],
        [0.239, 0.219, 0.144, 0.243, 0.295, 0.244, 0.278, 0.298, 0.174, 0.186, 0.266, 0.223, 0.245, 

In [8]:
# --- 7 · recall@1 ----------------------------------------------------------
# Does each frame's top-scoring caption happen to be the true one?
# Prediction written down BEFORE looking: 0.75
PREDICTION = 0.75

hits = (S.argmax(dim=1) == torch.arange(N))
print(f"prediction : {PREDICTION:.2f}")
print(f"recall@1   : {hits.float().mean():.2f}  ({hits.sum().item()}/{N})")
print(f"chance     : {1/N:.2f}")
print("misses     :", [i for i, h in enumerate(hits.tolist()) if not h])

prediction : 0.75
recall@1   : 0.40  (8/20)
chance     : 0.05
misses     : [0, 6, 7, 8, 10, 11, 12, 13, 14, 17, 18, 19]


In [9]:
# --- 8 · ranks, both directions --------------------------------------------
# recall@1 is all-or-nothing: a true caption ranked 2nd of 20 scores the same as
# one ranked last. Rank shows how badly the misses actually missed.
#
# Two directions, two different questions:
#   image->text : given a frame, find the words
#   text->image : given words, find the frame  <- the direction the product uses
def report(M, name):
    """M: rows = queries, cols = candidates, diagonal = the true match."""
    hits  = (M.argmax(dim=1) == torch.arange(N))
    ranks = (M > M.diag().unsqueeze(1)).sum(dim=1) + 1   # ties favour the truth
    print(f"{name:12s} recall@1 {hits.float().mean():.2f}  "
          f"median rank {ranks.float().median():.1f}  "
          f"(chance r@1 {1/N:.2f}, chance median rank {(N+1)/2:.1f})")
    return ranks

r_i2t = report(S,   "image->text")
r_t2i = report(S.T, "text->image")

print()
for i in range(N):
    print(f"{i:02d}  i->t rank {r_i2t[i]:2d} (won by caption {S[i].argmax():02d})   "
          f"t->i rank {r_t2i[i]:2d} (won by frame {S[:, i].argmax():02d})")

# Hubs: a caption that wins rows it does not own. Count them.
from collections import Counter
print("\ncaptions winning >1 row:",
      {k: v for k, v in Counter(S.argmax(dim=1).tolist()).items() if v > 1})

image->text  recall@1 0.40  median rank 4.0  (chance r@1 0.05, chance median rank 10.5)
text->image  recall@1 0.40  median rank 2.0  (chance r@1 0.05, chance median rank 10.5)

00  i->t rank  6 (won by caption 04)   t->i rank  8 (won by frame 01)
01  i->t rank  1 (won by caption 01)   t->i rank  1 (won by frame 01)
02  i->t rank  1 (won by caption 02)   t->i rank  1 (won by frame 02)
03  i->t rank  1 (won by caption 03)   t->i rank 11 (won by frame 17)
04  i->t rank  1 (won by caption 04)   t->i rank  7 (won by frame 06)
05  i->t rank  1 (won by caption 05)   t->i rank  1 (won by frame 05)
06  i->t rank  4 (won by caption 07)   t->i rank  1 (won by frame 06)
07  i->t rank  4 (won by caption 04)   t->i rank 18 (won by frame 06)
08  i->t rank 11 (won by caption 15)   t->i rank  4 (won by frame 11)
09  i->t rank  1 (won by caption 09)   t->i rank  2 (won by frame 10)
10  i->t rank 10 (won by caption 09)   t->i rank 12 (won by frame 18)
11  i->t rank  4 (won by caption 16)   t->i rank  1 (

In [10]:
# --- 9 · window sweep ------------------------------------------------------
# ±5s was chosen in advance, and it stays on the record. This is a sensitivity
# analysis, NOT a search for the best window: report the whole curve, do not swap
# in whichever value scored highest.
#
# Watch `max tok`. Once it reaches 77, CLIP truncates silently and the wider
# windows are not really wider — that is a ceiling, not a tuning knob.
print(f"{'window':>8}  {'r@1 i->t':>9}  {'med i->t':>9}  {'r@1 t->i':>9}  {'max tok':>8}")
for pad in (3.0, 5.0, 8.0, 12.0, 20.0):
    caps_p = [caption(p["ts"], pad=pad) for p in pairs]
    Sp = img @ encode_text(caps_p).T
    r1_i2t = (Sp.argmax(dim=1)   == torch.arange(N)).float().mean().item()
    r1_t2i = (Sp.T.argmax(dim=1) == torch.arange(N)).float().mean().item()
    med    = ((Sp > Sp.diag().unsqueeze(1)).sum(dim=1) + 1).float().median().item()
    mx     = max(len(x) for x in proc.tokenizer(caps_p)["input_ids"])
    print(f"{pad:>7.0f}s  {r1_i2t:>9.2f}  {med:>9.1f}  {r1_t2i:>9.2f}  {mx:>8d}")

  window   r@1 i->t   med i->t   r@1 t->i   max tok
      3s       0.25        4.0       0.40        46
      5s       0.40        4.0       0.40        65


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (82 > 77). Running this sequence through the model will result in indexing errors


      8s       0.35        2.0       0.35        83
     12s       0.35        2.0       0.40       114
     20s       0.30        2.0       0.40       190


In [11]:
import numpy as np

matched = S.diag().mean().item()

rng, means = np.random.default_rng(SEED), []
for _ in range(100):
    while True:
        perm = rng.permutation(N)
        if not (perm == np.arange(N)).any():   # derangement: no caption stays put
            break
    means.append(S[torch.arange(N), torch.tensor(perm)].mean().item())

mu, sd = float(np.mean(means)), float(np.std(means))
print(f"matched   {matched:.4f}")
print(f"shuffled  {mu:.4f} ± {sd:.4f}  (100 derangements)")
print(f"delta     {matched - mu:+.4f}   =  {(matched - mu) / sd:.1f} shuffle SDs")

matched   0.2673
shuffled  0.2310 ± 0.0038  (100 derangements)
delta     +0.0363   =  9.5 shuffle SDs


In [12]:
# --- 10 · cross-video control ---------------------------------------------
def parse_vtt(path):
    """Same rolling-cue dedupe as cell 3, reusable for any video."""
    out = []
    for c in webvtt.read(str(path)):
        txt = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", "", c.text).replace("\n", " ")).strip()
        if not txt:
            continue
        if out and (txt in out[-1][2] or out[-1][2].endswith(txt)):
            continue
        if out and txt.startswith(out[-1][2]):
            txt = txt[len(out[-1][2]):].strip()
            if not txt:
                continue
        out.append((c.start_in_seconds, c.end_in_seconds, txt))
    return out

other = parse_vtt(ROOT / "data/raw/podcast01/podcast01.en.vtt")

def caption_from(cue_list, t, pad=PAD):
    lo, hi = t - pad, t + pad
    return " ".join(x for s, e, x in cue_list if e >= lo and s <= hi).strip()

# five timestamps spread across the podcast, same ±5s rule
span = other[-1][1]
other_ts   = [span * f for f in (0.1, 0.3, 0.5, 0.7, 0.9)]
other_caps = [caption_from(other, t) for t in other_ts]
for t, c in zip(other_ts, other_caps):
    print(f"{int(t//60):02d}:{int(t%60):02d}  {c}\n")

X = img @ encode_text(other_caps).T          # 20 frames x 5 captions — no diagonal
cross = X.mean().item()

print(f"matched (same video)   {matched:.4f}")
print(f"shuffled (same video)  {mu:.4f} ± {sd:.4f}")
print(f"cross-video            {cross:.4f}")
print(f"\nmatched - cross        {matched - cross:+.4f}")
print(f"shuffled - cross       {mu - cross:+.4f}   = {(mu - cross)/sd:.1f} shuffle SDs")

02:24  recently apart from around here in dumre gallway obviously um up towards Lo Lan's always really good um so yeah beautiful scenery if you on a good day when

07:12  in those areas and as it moves out from there cooling down and potentially sinking in colder areas now in the really cold areas as well we have the formation of ice yes okay makes sense

12:01  reflected back into space you can actually see that at the poles we kicking out more heat than we should be doing we're kind of reflecting more and that's because our oceans and those environments redistribute a lot of that

16:49  if it's buyant whether it's something that is very natural so like floating seaweed and sarg gasm um something that's been released after a tropical storm or it's something man-made that

21:37  currents they must have been in and track rotations of those currents so right they've got to hear by this date I wonder when they'll come back around again and that would tell us about some of that g circula

In [14]:
# --- 11 · checkpoint comparison -------------------------------------------
from transformers import AutoModel, AutoProcessor

def embed(model_id, images, texts):
    """Return L2-normalised (image, text) embeddings in the model's shared space."""
    m = AutoModel.from_pretrained(model_id).eval()
    p = AutoProcessor.from_pretrained(model_id)
    siglip = "siglip" in model_id

    tok_kw = dict(padding="max_length" if siglip else True, truncation=True)
    with torch.no_grad():
        v = m.vision_model(**p(images=images, return_tensors="pt")).pooler_output
        t = m.text_model(**p(text=texts, return_tensors="pt", **tok_kw)).pooler_output
        if not siglip:                       # CLIP projects; SigLIP already is there
            v, t = m.visual_projection(v), m.text_projection(t)

    return (v / v.norm(dim=-1, keepdim=True),
            t / t.norm(dim=-1, keepdim=True), m, p)

def three_levels(model_id, n_shuffle=100):
    caps = [p["caption"] for p in pairs]
    v, t, m, p = embed(model_id, imgs, caps)
    M = v @ t.T

    matched = M.diag().mean().item()
    rng, means = np.random.default_rng(SEED), []
    for _ in range(n_shuffle):
        while True:
            perm = rng.permutation(N)
            if not (perm == np.arange(N)).any():
                break
        means.append(M[torch.arange(N), torch.tensor(perm)].mean().item())
    mu_, sd_ = float(np.mean(means)), float(np.std(means))

    tok_kw = dict(padding="max_length" if "siglip" in model_id else True, truncation=True)
    with torch.no_grad():
        ot = m.text_model(**p(text=other_caps, return_tensors="pt", **tok_kw)).pooler_output
        if "siglip" not in model_id:
            ot = m.text_projection(ot)
    ot = ot / ot.norm(dim=-1, keepdim=True)
    cross = (v @ ot.T).mean().item()

    r1_i2t = (M.argmax(dim=1)   == torch.arange(N)).float().mean().item()
    r1_t2i = (M.T.argmax(dim=1) == torch.arange(N)).float().mean().item()
    med    = ((M > M.diag().unsqueeze(1)).sum(dim=1) + 1).float().median().item()

    return dict(matched=matched, shuf=mu_, sd=sd_, cross=cross,
                sep_sd=(matched - mu_) / sd_, cross_sd=(mu_ - cross) / sd_,
                r1_i2t=r1_i2t, r1_t2i=r1_t2i, med=med)

for mid in ("openai/clip-vit-base-patch32",
            "google/siglip-base-patch16-224",
            "google/siglip2-base-patch16-224"):
    r = three_levels(mid)
    print(f"\n{mid}")
    print(f"  matched {r['matched']:.4f}   shuffled {r['shuf']:.4f} ± {r['sd']:.4f}   "
          f"cross {r['cross']:.4f}")
    print(f"  separation  {r['sep_sd']:5.1f} SD (matched vs shuffled)   "
          f"{r['cross_sd']:5.1f} SD (shuffled vs cross)")
    print(f"  recall@1  i→t {r['r1_i2t']:.2f}   t→i {r['r1_t2i']:.2f}   "
          f"median rank {r['med']:.1f}")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 18114.80it/s]



openai/clip-vit-base-patch32
  matched 0.2673   shuffled 0.2310 ± 0.0038   cross 0.1784
  separation    9.5 SD (matched vs shuffled)    13.8 SD (shuffled vs cross)
  recall@1  i→t 0.40   t→i 0.40   median rank 4.0


Loading weights: 100%|██████████| 408/408 [00:00<00:00, 5960.77it/s]



google/siglip-base-patch16-224
  matched 0.0727   shuffled 0.0366 ± 0.0039   cross -0.0102
  separation    9.3 SD (matched vs shuffled)    12.1 SD (shuffled vs cross)
  recall@1  i→t 0.25   t→i 0.45   median rank 5.0


Loading weights: 100%|██████████| 408/408 [00:00<00:00, 4585.15it/s]



google/siglip2-base-patch16-224
  matched 0.1257   shuffled 0.1045 ± 0.0029   cross 0.0406
  separation    7.2 SD (matched vs shuffled)    21.7 SD (shuffled vs cross)
  recall@1  i→t 0.30   t→i 0.25   median rank 3.0


In [15]:
# --- 12 · the 77-token wall ------------------------------------------------
chunk = caption(1200.0, pad=150.0)                    # ~5 minutes of speech
words = chunk.split()
print(f"{len(words)} words\n")

tk  = proc.tokenizer
ids = tk(chunk)["input_ids"]                          # no truncation: the true length
print(f"{len(ids)} tokens, cap is {tk.model_max_length}")

kept    = tk.decode(tk(chunk, truncation=True)["input_ids"], skip_special_tokens=True)
dropped = len(words) - len(kept.split())
print(f"\nSURVIVES : {kept}")
print(f"\nDISCARDED: {dropped} words ({dropped/len(words):.0%}) — silently, no warning\n")

# the proof: full chunk vs its truncated prefix
a = encode_text([chunk])[0]
b = encode_text([kept])[0]
print(f"cosine(full, first-77-tokens) = {(a @ b).item():.6f}")

873 words

1121 tokens, cap is 77

SURVIVES : more weight w 0 , which is called our bias . now , our bias is what can allow us to shift this this function up or down , right ? and i 'll show a graphical example of this in a second . right ? but the equation is still the same . we take every input , multiply by corresponding weights , add a bias , pass through our non -

DISCARDED: 800 words (92%) — silently, no warning

cosine(full, first-77-tokens) = 1.000000
